# Agent Workflow Memory: Reusing Learned Task Trajectories

## What is Agent Workflow Memory (AWM)?

**Agent Workflow Memory (AWM)** is a memory architecture where an agent remembers and reuses entire
**multi-step procedures** (sequences of tool calls) that previously solved a task successfully, rather
than remembering isolated facts (episodic/semantic memory) or standalone executable code skills
(as in Voyager-style skill libraries).

The core idea, inspired by the `35_agent_workflow_memory` concept in
[FareedKhan-dev/all-agentic-architectures](https://github.com/FareedKhan-dev/all-agentic-architectures)
(re-implemented here from scratch, not copied), is:

> If a task the agent just solved is *structurally similar* to a future task (same shape, different
> specific values), don't re-plan from scratch — retrieve the workflow **template** that worked before,
> **adapt** its placeholders to the new task, and execute it directly.

### High-level workflow

1. **Act** — run the agent on a task with a small ReAct-ish loop (pick tool -> call tool -> observe -> repeat).
2. **Log the trajectory** — record the exact sequence of tool calls (name + arguments) that led to success.
3. **Abstract** — an LLM call turns the concrete trajectory into a generalized workflow template, replacing
   specific values with placeholders (e.g. `"1. look up {entity}'s value  2. multiply by {factor}"`).
4. **Store** — save `{task_description, workflow_steps, success}` in a workflow memory store.
5. **Retrieve & adapt** — for a new task, search the store for the most structurally similar remembered
   task. If found, ask the LLM to *fill in* the template's placeholders for the new task instead of
   planning from a blank slate, then execute the filled-in steps directly.
6. **Fallback** — if no similar workflow exists, fall back to normal from-scratch ReAct planning.

### When to use AWM

- Agents that face **repeated task families** with the same shape but different concrete values
  (e.g. "look up X, then compute Y from it" across many different X/Y).
- Environments with a stable, small tool surface, where the *sequence* of tool calls (not the code)
  is the reusable unit.
- Systems where reducing planning latency/tokens on recurring task shapes matters (agentic copilots,
  workflow automation, repeated data-lookup-and-compute pipelines).

### Strengths

- **Faster & cheaper** on recurring task shapes — adapting a template skips exploratory planning.
- **More consistent** — a workflow that worked before is less likely to go off-track than a fresh plan.
- **Complementary** to episodic/semantic memory (facts) and skill libraries (code) — it captures the
  *procedure*, sitting one level of abstraction above both.

### Weaknesses

- Templates can **overfit** to the surface form of the first task seen — retrieval must judge structural
  similarity, not just keyword overlap, or it will mis-apply a workflow to a task it doesn't actually fit.
- Requires a **success signal** to know which trajectories are worth abstracting and storing.
- Doesn't help with genuinely **novel** task shapes — those always fall back to full planning.
- Templates can go **stale** if the tools or their semantics change.


## What We Are Going to Do

1. Define a tiny toolset: a `calculator` tool and a `lookup` tool over a small in-notebook dataset.
2. Build a minimal ReAct-ish agent loop that **logs every tool call it makes** (the "trajectory").
3. Run the agent on a couple of tasks, and after each **success**, use the LLM to **abstract** the
   concrete trajectory into a reusable **workflow template**, stored in a `workflow_memory` list.
4. Give the agent a **new, structurally similar task** — retrieve the closest matching template,
   have the LLM **adapt** it (fill in new placeholders), and execute the adapted steps directly,
   comparing this against solving the task by full planning.
5. Give the agent a **novel, unrelated task** — show that no workflow matches, so it falls back to
   normal from-scratch planning.

Throughout, we narrate what's happening so the speed/directness difference between "adapt a
remembered workflow" and "plan from scratch" is visible.


In [ ]:
# ============ IMPORTS & SETUP ============
import json
import re
from typing import Any

from helpers import get_llm

llm = get_llm()


## Defining a Small Toolset

We use two tools:

- **`lookup(entity)`** — looks up a numeric value for an entity in a small in-notebook "database"
  (stand-in for e.g. a product catalog, a stock price table, an inventory system).
- **`calculator(expression)`** — evaluates a simple arithmetic expression.

Structurally similar tasks will look like: *"look up {entity}'s value, then do some arithmetic with it"*.


In [ ]:
# ============ TOOLS ============
DATASET = {
    "widget_a": 12,
    "widget_b": 7,
    "widget_c": 25,
    "gadget_x": 40,
    "gadget_y": 3,
}


def lookup(entity: str) -> str:
    """Look up the numeric value associated with an entity in the dataset."""
    key = entity.strip().lower().replace(" ", "_")
    if key not in DATASET:
        return f"ERROR: '{entity}' not found in dataset."
    return str(DATASET[key])


def calculator(expression: str) -> str:
    """Evaluate a simple arithmetic expression, e.g. '12 * 4'."""
    # Only allow digits, operators, spaces, parentheses, and decimal points.
    if not re.fullmatch(r"[0-9+\-*/().\s]+", expression):
        return f"ERROR: unsafe expression '{expression}'."
    try:
        return str(eval(expression))  # noqa: S307 - sandboxed to arithmetic chars above
    except Exception as exc:
        return f"ERROR: {exc}"


TOOLS = {
    "lookup": lookup,
    "calculator": calculator,
}

TOOL_DESCRIPTIONS = """\
- lookup(entity: str) -> str : look up a numeric value for an entity in the dataset
- calculator(expression: str) -> str : evaluate an arithmetic expression
"""


## The Workflow Memory Store

Each record is `{task_description, workflow_steps, success}`, where `workflow_steps` is a list of
human-readable, **generalized** step strings (placeholders like `{entity}`, `{factor}` instead of
concrete values). This is intentionally simple — a list in memory — to keep the retrieval logic
transparent; a production system might back this with a vector store keyed on `task_description`.


In [ ]:
# ============ WORKFLOW MEMORY STORE ============
workflow_memory: list[dict[str, Any]] = []


def add_workflow(task_description: str, workflow_steps: list[str], success: bool) -> None:
    workflow_memory.append(
        {
            "task_description": task_description,
            "workflow_steps": workflow_steps,
            "success": success,
        }
    )


def show_workflow_memory() -> None:
    for i, wf in enumerate(workflow_memory):
        print(f"[{i}] task: {wf['task_description']!r}  success={wf['success']}")
        for step in wf["workflow_steps"]:
            print(f"      - {step}")


## A Minimal ReAct-ish Agent Loop (with Trajectory Logging)

This is a deliberately simple loop: at each step we ask the LLM to either call a tool (as JSON) or
declare `"final_answer"`. Every tool call the agent actually makes is appended to `trajectory` —
this concrete, executed sequence is what we'll later abstract into a workflow template.


In [ ]:
# ============ REACT-ISH LOOP WITH TRAJECTORY LOGGING ============
REACT_SYSTEM_PROMPT = f"""You are a task-solving agent with access to these tools:
{TOOL_DESCRIPTIONS}
At each turn, respond with ONLY a JSON object, no extra text, in one of these two forms:
1. To call a tool:      {{"action": "tool", "name": "<tool_name>", "args": {{...}}}}
2. To give final answer: {{"action": "final_answer", "value": "<answer>"}}

Use the tool observations you've been given so far to decide your next step.
Keep the trajectory as short as possible.
"""


def run_agent(task: str, max_steps: int = 5) -> tuple[str, list[dict]]:
    """Run the ReAct-ish loop from scratch (blank-slate planning). Returns (final_answer, trajectory)."""
    trajectory: list[dict] = []
    history = f"Task: {task}\n"
    for _ in range(max_steps):
        response = llm.invoke(
            [
                {"role": "system", "content": REACT_SYSTEM_PROMPT},
                {"role": "user", "content": history},
            ]
        )
        raw = response.content.strip()
        try:
            step = json.loads(raw)
        except json.JSONDecodeError:
            history += f"\n(Could not parse response: {raw!r}. Respond with valid JSON only.)\n"
            continue

        if step.get("action") == "final_answer":
            return step.get("value", ""), trajectory

        if step.get("action") == "tool":
            name, args = step.get("name"), step.get("args", {})
            tool_fn = TOOLS.get(name)
            if tool_fn is None:
                observation = f"ERROR: unknown tool '{name}'"
            else:
                observation = tool_fn(**args)
            trajectory.append({"tool": name, "args": args, "observation": observation})
            history += f"\nCalled {name}({args}) -> {observation}\n"

    return "FAILED: max steps reached", trajectory


## Abstracting a Trajectory into a Reusable Workflow Template

Once a task succeeds, we ask the LLM to look at the **concrete** trajectory and rewrite it as a
**generalized** step list, replacing specific values (entity names, numbers) with placeholders like
`{entity}` and `{factor}`. This generalized template is what gets stored in `workflow_memory` — not
the raw trajectory.


In [ ]:
# ============ ABSTRACTION: TRAJECTORY -> GENERALIZED WORKFLOW TEMPLATE ============
ABSTRACTION_PROMPT = """You just watched an agent solve a task using this sequence of tool calls:

Task: {task}

Trajectory:
{trajectory}

Rewrite this as a SHORT, GENERALIZED workflow template: a numbered list of steps describing the
PROCEDURE in the abstract, replacing concrete values (entity names, numbers, factors) with
placeholders like {{entity}}, {{factor}}, {{amount}}. Do not include the concrete values themselves.

Respond with ONLY a JSON list of strings, e.g.:
["1. look up {{entity}}'s value", "2. multiply the result by {{factor}}"]
"""


def abstract_trajectory(task: str, trajectory: list[dict]) -> list[str]:
    trajectory_str = "\n".join(
        f"- {t['tool']}({t['args']}) -> {t['observation']}" for t in trajectory
    )
    response = llm.invoke(
        [{"role": "user", "content": ABSTRACTION_PROMPT.format(task=task, trajectory=trajectory_str)}]
    )
    raw = response.content.strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        # Fall back to a naive split if the LLM didn't return clean JSON.
        return [line.strip("- ") for line in raw.splitlines() if line.strip()]


In [ ]:
# ============ RUN THE AGENT ON A FIRST TASK & LEARN A WORKFLOW ============
task_1 = "Look up widget_a's value and multiply it by 3."
answer_1, trajectory_1 = run_agent(task_1)

print("Task:", task_1)
print("Answer:", answer_1)
print("Trajectory:", trajectory_1)


In [ ]:
success_1 = answer_1 not in ("", None) and not str(answer_1).startswith("FAILED")

if success_1:
    template_1 = abstract_trajectory(task_1, trajectory_1)
    add_workflow(task_1, template_1, success=True)

show_workflow_memory()


### Discussion of the Output

The agent solved `task_1` from a blank slate: two tool calls (`lookup` then `calculator`), each
decided one step at a time by the LLM re-reasoning about the whole history so far. After success,
the abstraction step compressed that concrete trajectory into a generalized 2-step template with
`{entity}` and `{factor}` placeholders. That template — not the raw trajectory — is what now lives
in `workflow_memory`, ready to be reused for a differently-worded but structurally identical task.


In [ ]:
# ============ LEARN A SECOND, DIFFERENTLY-SHAPED WORKFLOW ============
task_2 = "Look up gadget_x's value, then look up gadget_y's value, and add them together."
answer_2, trajectory_2 = run_agent(task_2)

print("Task:", task_2)
print("Answer:", answer_2)
print("Trajectory:", trajectory_2)

success_2 = answer_2 not in ("", None) and not str(answer_2).startswith("FAILED")
if success_2:
    template_2 = abstract_trajectory(task_2, trajectory_2)
    add_workflow(task_2, template_2, success=True)

show_workflow_memory()


We now have **two** distinct workflow templates in memory:

1. `lookup(entity) -> calculator(value * factor)` — a "look up then scale" shape.
2. `lookup(entity_1) -> lookup(entity_2) -> calculator(sum)` — a "look up two things then combine" shape.

This diversity is what makes retrieval meaningful in the next step: a new task needs to be matched
against the *right* template, not just any template.


## Retrieving the Most Relevant Remembered Workflow

For a new task, we ask the LLM to judge which stored `task_description` (if any) is **structurally
similar** to the new task — same shape of steps, regardless of the specific entities/numbers involved.
This is a lightweight LLM-similarity match rather than embedding search, which keeps the notebook
self-contained, but a production system would typically embed `task_description` and use a vector
store for this lookup.


In [ ]:
# ============ RETRIEVE THE CLOSEST MATCHING WORKFLOW ============
RETRIEVAL_PROMPT = """You are matching a new task against a library of remembered workflows, by
STRUCTURE (the shape/sequence of steps), not by the specific values mentioned.

New task: {new_task}

Remembered workflows:
{workflows}

If one of the remembered workflows has the SAME STRUCTURE as the new task (same kind of steps in the
same order, just with different specific values), respond with ONLY its index number (e.g. "0").
If none of them match structurally, respond with ONLY the word "NONE".
"""


def retrieve_workflow(new_task: str) -> dict | None:
    if not workflow_memory:
        return None
    workflows_str = "\n".join(
        f"[{i}] {wf['task_description']} -> steps: {wf['workflow_steps']}"
        for i, wf in enumerate(workflow_memory)
    )
    response = llm.invoke(
        [{"role": "user", "content": RETRIEVAL_PROMPT.format(new_task=new_task, workflows=workflows_str)}]
    )
    raw = response.content.strip()
    if raw.upper().startswith("NONE"):
        return None
    try:
        idx = int(re.search(r"\d+", raw).group())
        return workflow_memory[idx]
    except (AttributeError, ValueError, IndexError):
        return None


## Adapting a Retrieved Workflow to a New Task

Instead of planning from scratch, we hand the LLM the **retrieved template** and the **new task**,
and ask it to fill in the placeholders with concrete values and produce concrete tool calls to
execute — directly, without the step-by-step exploratory reasoning `run_agent` does.


In [ ]:
# ============ ADAPT & EXECUTE A RETRIEVED WORKFLOW ============
ADAPTATION_PROMPT = f"""You are given a generalized workflow template and a new task. Fill in the
template's placeholders with concrete values from the new task, and produce the CONCRETE sequence of
tool calls needed to execute it.

Available tools:
{TOOL_DESCRIPTIONS}
Template steps: {{template}}
New task: {{new_task}}

Respond with ONLY a JSON list of tool calls, e.g.:
[{{"name": "lookup", "args": {{"entity": "widget_b"}}}}, {{"name": "calculator", "args": {{"expression": "7 * 2"}}}}]

Note: for a calculator step, you must substitute in the ACTUAL numeric value returned by any prior
lookup step in this same plan (use the dataset values you can infer/reason about if needed).
"""


def adapt_and_execute(template_workflow: dict, new_task: str) -> tuple[str, list[dict]]:
    response = llm.invoke(
        [
            {
                "role": "user",
                "content": ADAPTATION_PROMPT.format(
                    template=template_workflow["workflow_steps"], new_task=new_task
                ),
            }
        ]
    )
    raw = response.content.strip()
    try:
        plan = json.loads(raw)
    except json.JSONDecodeError:
        plan = []

    trajectory = []
    last_observation = None
    for step in plan:
        name, args = step.get("name"), step.get("args", {})
        tool_fn = TOOLS.get(name)
        observation = tool_fn(**args) if tool_fn else f"ERROR: unknown tool '{name}'"
        trajectory.append({"tool": name, "args": args, "observation": observation})
        last_observation = observation

    return last_observation, trajectory


In [ ]:
# ============ NEW, STRUCTURALLY-SIMILAR TASK: ADAPT INSTEAD OF PLAN ============
new_similar_task = "Look up widget_c's value and multiply it by 4."

retrieved = retrieve_workflow(new_similar_task)
print("Retrieved template:", retrieved["workflow_steps"] if retrieved else None)

if retrieved:
    answer_adapted, trajectory_adapted = adapt_and_execute(retrieved, new_similar_task)
    print("\n--- Solved by ADAPTING a remembered workflow (no blank-slate planning) ---")
    print("Task:", new_similar_task)
    print("Trajectory:", trajectory_adapted)
    print("Answer:", answer_adapted)
else:
    print("No matching workflow found - would fall back to run_agent().")


### Discussion of the Output

Compare this to how `task_1` was solved earlier: there, the agent took multiple LLM calls, each one
re-reading the whole running history and deciding the *next single step*. Here, retrieval matched
`new_similar_task` to template `[0]` (the "look up then scale" shape) in one LLM call, and
`adapt_and_execute` produced the **entire concrete tool-call plan in a single shot**, then executed
it directly — no exploratory step-by-step reasoning needed. This is the speed/directness gain AWM is
meant to provide on recurring task shapes: fewer LLM round-trips, and a plan that mirrors a
previously-successful procedure instead of being re-derived from zero.


## A Novel, Unrelated Task: No Matching Workflow, Fall Back to Planning

Now let's give the agent a task with a genuinely different shape — one that doesn't match either
stored template. Retrieval should return `None`, and we fall back to full `run_agent` planning,
exactly as we did for `task_1` and `task_2` before anything was in memory.


In [ ]:
# ============ NOVEL TASK: NO WORKFLOW MATCH -> FALL BACK TO FROM-SCRATCH PLANNING ============
novel_task = "What is 17 times 5, minus 9?"

retrieved_novel = retrieve_workflow(novel_task)
print("Retrieved template for novel task:", retrieved_novel)

if retrieved_novel is None:
    print("\n--- No matching workflow: falling back to blank-slate ReAct planning ---")
    answer_novel, trajectory_novel = run_agent(novel_task)
    print("Task:", novel_task)
    print("Trajectory:", trajectory_novel)
    print("Answer:", answer_novel)


### Discussion of the Output

`novel_task` doesn't involve a `lookup` at all — it's pure arithmetic, structurally unlike either
remembered workflow (which both start with one or more `lookup` calls). Retrieval correctly returns
`None`, and the agent falls back to the same step-by-step `run_agent` loop used before any workflow
memory existed. This is the expected trade-off: AWM accelerates *recurring* task shapes, but a
genuinely novel shape still requires full planning — and, if it succeeds, could itself be abstracted
into a *third* workflow template for next time.


## Summary & Key Takeaways

- **Agent Workflow Memory (AWM)** stores generalized **procedures** — sequences of tool calls that
  solved past tasks — as `{task_description, workflow_steps, success}` records, distinct from
  remembering individual facts or standalone code.
- The pipeline has four moving parts: **act & log trajectory** -> **abstract into a template** ->
  **retrieve** the closest structural match for a new task -> **adapt & execute** the template
  directly, skipping blank-slate planning.
- Retrieval must judge **structural similarity** (same shape of steps), not surface/keyword overlap —
  we used a lightweight LLM-similarity check here; a production system would typically embed
  `task_description` and use a vector store instead.
- When no workflow matches (a genuinely novel task), the agent **falls back to normal from-scratch
  ReAct planning** — AWM only pays off on task shapes the agent has seen before.
- **How this differs from Voyager-style skill libraries:** Voyager stores standalone, directly
  executable **code skills** (e.g. a JavaScript function to "build a house") that can be called like
  black-box subroutines. AWM instead stores **procedural knowledge about which tools to call, in
  which order, for which kind of task** — a workflow template still needs the agent's tool-calling
  loop to execute it (each step is adapted and dispatched to a real tool), rather than being an
  opaque, self-contained callable. AWM is one level closer to "how the agent should plan," where
  Voyager is one level closer to "new capabilities the agent can invoke."

This notebook is one of several memory-architecture notebooks under
`07_Advanced_Agentic_Systems/Memory_and_State/Agentic_Memory_Architectures/` — its siblings (authored
separately) cover **Graph Memory**, **MemGPT-style tiered memory**, and a **Voyager-style skill
library**. Together they illustrate a spectrum of what an agent can remember: facts, graph-structured
relationships, tiered context, standalone code skills, and — here — reusable task-solving workflows.
